# Predictive Circuit Coding Training

This notebook uses the shared Drive dataset as a read-only source, keeps artifacts local during the run for reliability, and exports the finished run to a clearly separate Drive folder when training is done.

Default behavior is simple: use the full canonical dataset unless you explicitly turn subset selection back on.

In [ ]:
from google.colab import drive
from pathlib import Path
import shutil
from datetime import datetime

drive.mount('/content/drive')
repo_root = Path('/content/predictive_circuit_coding')
github_repo_url = 'https://github.com/jacobposchl/predictive_circuit_coding'

# Shared folder from the storage-heavy Drive account: read dataset from here.
shared_drive_root = Path('/content/drive/MyDrive/pcc_runs')
drive_data_root = shared_drive_root / 'data' / 'allen_visual_behavior_neuropixels'

# Keep exported Colab outputs in a clearly different folder name so they do not get
# confused with the source dataset bundle.
drive_export_root = Path('/content/drive/MyDrive/pcc_colab_outputs')
drive_export_root.mkdir(parents=True, exist_ok=True)

print('Repo root:', repo_root)
print('GitHub repo:', github_repo_url)
print('Shared dataset root:', drive_data_root)
print('Drive export root:', drive_export_root)

In [ ]:
if not repo_root.exists():
    !git clone {github_repo_url} {repo_root}
%cd {repo_root}
# Keep Colab installs minimal to avoid restart prompts from preloaded widget packages.
!pip install -e .

In [ ]:
repo_dataset_root = repo_root / 'data' / 'allen_visual_behavior_neuropixels'
repo_artifacts_root = repo_root / 'artifacts'

assert drive_data_root.exists(), f'Missing Drive dataset bundle: {drive_data_root}'

repo_dataset_root.parent.mkdir(parents=True, exist_ok=True)
if repo_dataset_root.exists() or repo_dataset_root.is_symlink():
    if repo_dataset_root.is_symlink():
        repo_dataset_root.unlink()
    else:
        shutil.rmtree(repo_dataset_root)
repo_dataset_root.symlink_to(drive_data_root, target_is_directory=True)

if repo_artifacts_root.exists():
    shutil.rmtree(repo_artifacts_root)
repo_artifacts_root.mkdir(parents=True, exist_ok=True)

print('Linked dataset into repo:', repo_dataset_root)
print('Using local artifact directory:', repo_artifacts_root)

In [ ]:
import os
import subprocess
import yaml
from predictive_circuit_coding.utils import NotebookStageReporter, verify_paths_exist

def run_and_stream(command, *, cwd=None):
    env = dict(os.environ)
    env['PYTHONUNBUFFERED'] = '1'
    process = subprocess.Popen(
        command,
        cwd=str(cwd) if cwd else None,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
        env=env,
    )
    assert process.stdout is not None
    for line in process.stdout:
        print(line, end='')
    return_code = process.wait()
    if return_code != 0:
        raise subprocess.CalledProcessError(return_code, command)
    return return_code

reporter = NotebookStageReporter(name='colab-train', expected_duration='2-10 minutes for debug runs, longer for full A100 runs')
reporter.banner('Predictive Circuit Coding Training', subtitle='Setup, preflight, runtime selection, training, evaluation, and export')

base_experiment_config = repo_root / 'configs' / 'pcc' / 'predictive_circuit_coding_base.yaml'
data_config = repo_root / 'configs' / 'pcc' / 'allen_visual_behavior_neuropixels_local.yaml'
runtime_experiment_config = repo_root / 'artifacts' / 'colab_runtime_experiment.yaml'
checkpoint_dir = repo_root / 'artifacts' / 'checkpoints'
resume_checkpoint = checkpoint_dir / 'pcc_best.pt'

# Default notebook mode: small familiar-session subset for faster concept checks.
USE_FULL_DATASET = False
TARGET_EXPERIENCE_LEVEL = 'Familiar'
MAX_SESSIONS = 10

selection_disabled = {
    'output_name': 'full_dataset_colab',
    'session_ids': [],
    'subject_ids': [],
    'exclude_session_ids': [],
    'exclude_subject_ids': [],
    'session_ids_file': None,
    'subject_ids_file': None,
    'exclude_session_ids_file': None,
    'exclude_subject_ids_file': None,
    'experience_levels': [],
    'session_types': [],
    'image_sets': [],
    'session_numbers': [],
    'project_codes': [],
    'brain_regions_any': [],
    'min_n_units': None,
    'max_n_units': None,
    'min_trial_count': None,
    'max_trial_count': None,
    'min_duration_s': None,
    'max_duration_s': None,
    'split_seed': None,
    'split_primary_axis': None,
    'train_fraction': None,
    'valid_fraction': None,
    'discovery_fraction': None,
    'test_fraction': None,
}

experiment_payload = yaml.safe_load(base_experiment_config.read_text(encoding='utf-8'))
catalog_csv = repo_root / 'data' / 'allen_visual_behavior_neuropixels' / 'manifests' / 'session_catalog.csv'
if USE_FULL_DATASET:
    experiment_payload['dataset_selection'] = selection_disabled
else:
    import pandas as pd
    catalog = pd.read_csv(catalog_csv)
    subset = catalog.loc[catalog['experience_level'] == TARGET_EXPERIENCE_LEVEL].sort_values('session_id').head(MAX_SESSIONS)
    assert len(subset) > 0, f'No sessions found for experience_level={TARGET_EXPERIENCE_LEVEL}'
    session_ids_file = repo_root / 'artifacts' / f'{TARGET_EXPERIENCE_LEVEL.lower()}_{MAX_SESSIONS}_session_ids.txt'
    session_ids_file.write_text('\n'.join(subset['session_id'].astype(str).tolist()) + '\n', encoding='utf-8')
    experiment_payload['dataset_selection'] = {
        'output_name': f'{TARGET_EXPERIENCE_LEVEL.lower()}_{MAX_SESSIONS}_session_subset',
        'session_ids': [],
        'subject_ids': [],
        'exclude_session_ids': [],
        'exclude_subject_ids': [],
        'session_ids_file': str(session_ids_file),
        'subject_ids_file': None,
        'exclude_session_ids_file': None,
        'exclude_subject_ids_file': None,
        'experience_levels': [],
        'session_types': [],
        'image_sets': [],
        'session_numbers': [],
        'project_codes': [],
        'brain_regions_any': [],
        'min_n_units': None,
        'max_n_units': None,
        'min_trial_count': None,
        'max_trial_count': None,
        'min_duration_s': None,
        'max_duration_s': None,
        'split_seed': 7,
        'split_primary_axis': 'session',
        'train_fraction': 0.6,
        'valid_fraction': 0.2,
        'discovery_fraction': 0.1,
        'test_fraction': 0.1,
    }

# For notebook runs, show progress frequently so the run feels alive.
experiment_payload.setdefault('training', {})['log_every_steps'] = 1

runtime_experiment_config.write_text(yaml.safe_dump(experiment_payload, sort_keys=False), encoding='utf-8')
experiment_config = runtime_experiment_config

dataset_selection_active = any(
    value not in (None, [], '', {})
    for key, value in experiment_payload.get('dataset_selection', {}).items()
    if key != 'output_name'
)

run_label = datetime.now().strftime('train_run_%Y%m%d_%H%M%S')
drive_run_export_root = drive_export_root / run_label

print('Experiment config:', experiment_config)
print('Data config:', data_config)
print('Runtime selection active:', dataset_selection_active)
if not USE_FULL_DATASET:
    print('Target experience level:', TARGET_EXPERIENCE_LEVEL)
    print('Max sessions:', MAX_SESSIONS)
print('Checkpoint path:', resume_checkpoint)
print('Drive export path:', drive_run_export_root)

In [ ]:
reporter.begin('preflight', next_artifact='best checkpoint + evaluation summary')
paths_ok = verify_paths_exist({
    'experiment_config': experiment_config,
    'data_config': data_config,
    'drive_dataset_root': drive_data_root,
})
print(paths_ok)
assert all(paths_ok.values()), 'Missing config files or dataset bundle for training notebook.'

if dataset_selection_active:
    reporter.begin('runtime selection', next_artifact='selected split manifest')
    run_and_stream([
        'pcc-prepare-data',
        'materialize-runtime-selection',
        '--config', str(experiment_config),
        '--data-config', str(data_config),
    ], cwd=repo_root)
    reporter.finish('runtime selection')
else:
    print('Using the full canonical dataset. No runtime subset will be materialized.')

reporter.finish('preflight')

In [ ]:
reporter.begin('training', next_artifact='local checkpoint under artifacts/checkpoints')
%cd {repo_root}
run_and_stream([
    'pcc-train',
    '--config', str(experiment_config),
    '--data-config', str(data_config),
], cwd=repo_root)
if not resume_checkpoint.exists():
    checkpoint_candidates = sorted(checkpoint_dir.glob('*.pt'))
    print('Available checkpoints:', [path.name for path in checkpoint_candidates])
    raise AssertionError(f'Expected checkpoint was not written: {resume_checkpoint}')
reporter.note_checkpoint(resume_checkpoint)
reporter.finish('training')

In [ ]:
reporter.begin('evaluation', next_artifact='local evaluation summary json')
%cd {repo_root}
run_and_stream([
    'pcc-evaluate',
    '--config', str(experiment_config),
    '--data-config', str(data_config),
    '--checkpoint', str(resume_checkpoint),
    '--split', 'valid',
], cwd=repo_root)
reporter.finish('evaluation')

In [ ]:
reporter.begin('export', next_artifact='Drive copy of local artifacts')
if drive_run_export_root.exists():
    shutil.rmtree(drive_run_export_root)
shutil.copytree(repo_artifacts_root, drive_run_export_root)
print('Exported local artifacts to:', drive_run_export_root)
reporter.finish('export')